# Superpave PG Binder Selector
**Prepared by Abenezer Tilahun** · AASHTO M320 / M323 · SHRP (high) and LTPP (low) models · Reliability 98 %

Upload daily temperature data. The notebook gives:

1. **Annual 7-day consecutive maximum temperature**: table and graph
2. **Annual minimum temperature**: table and graph
3. **Monthly temperature heatmaps** (Tmax and Tmin)
4. **Final PG grade** and its **specification table**
5. **Binder type** (crude oil / high-quality crude oil / modifier required) and the **laboratory tests**

**How to use it**
1. Click **Runtime → Run all** (or press ▶ on each cell from top to bottom).
2. Fill in the **Step 1** form. Leave *latitude* empty to use the latitude in your file (e.g. the GEOGR2 column).
3. **Step 2** shows a **Choose Files** button: pick your CSV or Excel file.
4. The results appear under **Step 3**. To change ESAL, speed or the other inputs, edit Step 1, run Step 1 again, then run **Step 3** again. No need to upload again.

Files in many layouts are read automatically, for example *Name, element (TMPMAX/TMPMIN), Year, Month, 1…31* or *Date, Tmax, Tmin*.
The data checks and gap filling run in the background and are not shown.


In [ ]:
#@title ⚙️ Load the calculation engine (run this first; no need to open it) { display-mode: "form" }
import os

# Superpave PG Binder Selector: Python version of engine/ and parser/ (same rules, same messages).
# Prepared by Abenezer Tilahun. Engine version 1.3.0. Reliability is fixed at 98 %.
#
# Keep this file identical in behaviour to the TypeScript packages; colab/parity_check.py compares them.

import csv
import io
import math
import re
import unicodedata
from datetime import date, datetime, timedelta

ENGINE_VERSION = '1.3.0'  # 1.1.0: >= 75% days per year; 1.2.0: heatmap means, low T_air = mean; 1.3.0: binder type

# ───────────────────────────── errors ─────────────────────────────


class AnalysisError(Exception):
    """A problem with the data or inputs; str(error) is one plain sentence for the user."""

    def __init__(self, code, message):
        super().__init__(message)
        self.code = code
        self.message = message


def file_error(message):
    return AnalysisError('BAD_FILE', message)


# ───────────────────────────── dates ─────────────────────────────

_EPOCH = date(1970, 1, 1).toordinal()


def is_leap_year(y):
    return (y % 4 == 0 and y % 100 != 0) or y % 400 == 0


def days_in_month(y, m):
    return [31, 29 if is_leap_year(y) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][m - 1]


def days_in_year(y):
    return 366 if is_leap_year(y) else 365


def _is_int(v):
    return isinstance(v, int) and not isinstance(v, bool)


def is_valid_date(y, m, d):
    return (_is_int(y) and _is_int(m) and _is_int(d) and 1800 <= y <= 2200 and 1 <= m <= 12
            and 1 <= d <= days_in_month(y, m))


def to_day_number(y, m, d):
    return date(y, m, d).toordinal() - _EPOCH


def from_day_number(n):
    d = date.fromordinal(n + _EPOCH)
    return {'year': d.year, 'month': d.month, 'day': d.day}


def format_iso_date(n):
    return date.fromordinal(n + _EPOCH).isoformat()


# ───────────────────────────── config ─────────────────────────────

HIGH_GRADES = [46, 52, 58, 64, 70, 76, 82, 88]
LOW_GRADES = [-10, -16, -22, -28, -34, -40, -46]
SPEC_TABLE_LOW_GRADES = [-10, -16, -22, -28, -34, -40]
M320_LOW_GRADES = {
    46: [-34, -40, -46],
    52: [-10, -16, -22, -28, -34, -40, -46],
    58: [-16, -22, -28, -34, -40],
    64: [-10, -16, -22, -28, -34, -40],
    70: [-10, -16, -22, -28, -34, -40],
    76: [-10, -16, -22, -28, -34],
    82: [-10, -16, -22, -28, -34],
    88: [],
}
SPEC_LIMITS = {
    'flashPointMinC': 230, 'viscosityMaxPaS': 3, 'viscosityTestTempC': 135, 'originalDsrMinKPa': 1.0,
    'rtfoMassChangeMaxPct': 1.0, 'rtfoDsrMinKPa': 2.2, 'pavAgingHours': 20, 'pavPressureMPa': 2.1,
    'pavDsrMaxKPa': 5000, 'bbrStiffnessMaxMPa': 300, 'bbrMValueMin': 0.3, 'dttFailureStrainMinPct': 1.0,
}


def pav_aging_temperature_label(high):
    if high <= 52:
        return '90'
    if high <= 64:
        return '100'
    return '100 (110)'


def grade_label(high, low):
    return f'PG {high}{low}'


# AASHTO M323 Table 1: grade steps to add, or 'consider' (+1 only if accepted).
SPEED_CLASS_LIMITS = {'standingBelowKmh': 20, 'slowUpToKmh': 70}
SPEED_CLASS_LABELS = {'standing': 'Standing (< 20 km/h)', 'slow': 'Slow (20–70 km/h)', 'standard': 'Standard (> 70 km/h)'}
ESAL_CLASSES = [('lt0.3', '< 0.3', 0), ('0.3to3', '0.3 to < 3', 0.3), ('3to10', '3 to < 10', 3),
                ('10to30', '10 to < 30', 10), ('ge30', '≥ 30', 30)]
GRADE_BUMP_TABLE = {
    'lt0.3': {'standing': 'consider', 'slow': 0, 'standard': 0},
    '0.3to3': {'standing': 2, 'slow': 1, 'standard': 0},
    '3to10': {'standing': 2, 'slow': 1, 'standard': 0},
    '10to30': {'standing': 2, 'slow': 1, 'standard': 'consider'},
    'ge30': {'standing': 2, 'slow': 1, 'standard': 1},
}
GRADE_STEP_C = 6

# ───────────────────────────── cleaning and gap filling ─────────────────────────────

ELEMENTS = ('tmax', 'tmin')
TEMP_LIMITS = (-50, 60)
SHORT_GAP_MAX_DAYS = 3


def _finite(v):
    return v is not None and not isinstance(v, bool) and isinstance(v, (int, float)) and math.isfinite(v)


def clean_observations(observations):
    quality = {
        'rowsRead': len(observations), 'invalidDates': 0,
        'duplicates': {'tmax': 0, 'tmin': 0}, 'outOfRange': {'tmax': 0, 'tmin': 0}, 'tmaxBelowTmin': 0,
        'realDays': {'tmax': 0, 'tmin': 0}, 'interpolatedDays': {'tmax': 0, 'tmin': 0},
        'monthlyMeanDays': {'tmax': 0, 'tmin': 0},
    }
    by_day = {'tmax': {}, 'tmin': {}}
    for obs in observations:
        y, m, d = obs.get('year'), obs.get('month'), obs.get('day')
        if not is_valid_date(y, m, d):
            quality['invalidDates'] += 1
            continue
        dn = to_day_number(y, m, d)
        for el in ELEMENTS:
            if el not in obs:
                continue
            raw = obs[el]
            mp = by_day[el]
            if dn in mp:
                quality['duplicates'][el] += 1
                if mp[dn] is not None:
                    continue
            value = float(raw) if _finite(raw) else None
            if value is not None and (value < TEMP_LIMITS[0] or value > TEMP_LIMITS[1]):
                quality['outOfRange'][el] += 1
                value = None
            mp[dn] = value
    for dn, tmax in list(by_day['tmax'].items()):
        tmin = by_day['tmin'].get(dn)
        if tmax is not None and tmin is not None and tmax < tmin:
            by_day['tmax'][dn] = None
            by_day['tmin'][dn] = None
            quality['tmaxBelowTmin'] += 1
    series = {el: _build_series(el, by_day[el]) for el in ELEMENTS}
    for el in ELEMENTS:
        for kind in series[el]['kinds']:
            if kind == 'real':
                quality['realDays'][el] += 1
            elif kind == 'interpolated':
                quality['interpolatedDays'][el] += 1
            elif kind == 'monthlyMean':
                quality['monthlyMeanDays'][el] += 1
    return series, quality


def _build_series(element, mp):
    days = [d for d, v in mp.items() if v is not None]
    if not days:
        return {'element': element, 'startDay': 0, 'values': [], 'kinds': []}
    start, end = min(days), max(days)
    n = end - start + 1
    values = [None] * n
    kinds = ['missing'] * n
    for d, v in mp.items():
        if v is None:
            continue
        values[d - start] = v
        kinds[d - start] = 'real'
    _fill_gaps(start, values, kinds)
    return {'element': element, 'startDay': start, 'values': values, 'kinds': kinds}


def _monthly_means(start, values, kinds):
    s = [0.0] * 13
    c = [0] * 13
    for i, v in enumerate(values):
        if kinds[i] != 'real' or v is None:
            continue
        m = from_day_number(start + i)['month']
        s[m] += v
        c[m] += 1
    return [s[m] / c[m] if c[m] > 0 else None for m in range(13)]


def _fill_gaps(start, values, kinds):
    means = _monthly_means(start, values, kinds)
    i = 0
    n = len(values)
    while i < n:
        if kinds[i] != 'missing':
            i += 1
            continue
        j = i
        while j + 1 < n and kinds[j + 1] == 'missing':
            j += 1
        gap = j - i + 1
        if gap <= SHORT_GAP_MAX_DAYS:
            before, after = values[i - 1], values[j + 1]
            for k in range(i, j + 1):
                values[k] = before + ((after - before) * (k - i + 1)) / (gap + 1)
                kinds[k] = 'interpolated'
        else:
            for k in range(i, j + 1):
                mean = means[from_day_number(start + k)['month']]
                if mean is None:
                    continue
                values[k] = mean
                kinds[k] = 'monthlyMean'
        i = j + 1


def year_coverage(series):
    if not series['values']:
        return []
    real = {}
    for i, kind in enumerate(series['kinds']):
        if kind != 'real':
            continue
        y = from_day_number(series['startDay'] + i)['year']
        real[y] = real.get(y, 0) + 1
    first = from_day_number(series['startDay'])['year']
    last = from_day_number(series['startDay'] + len(series['values']) - 1)['year']
    return [{'year': y, 'realDays': real.get(y, 0), 'daysInYear': days_in_year(y),
             'realFraction': real.get(y, 0) / days_in_year(y)} for y in range(first, last + 1)]


# ───────────────────────────── yearly extremes ─────────────────────────────

WINDOW_DAYS = 7


def _usable_value(series, dn):
    i = dn - series['startDay']
    if i < 0 or i >= len(series['values']) or series['kinds'][i] not in ('real', 'interpolated'):
        return None
    return series['values'][i]


def seven_day_max_for_year(series, year):
    first, last = to_day_number(year, 1, 1), to_day_number(year, 12, 31)
    best = None
    start = first
    while start + WINDOW_DAYS - 1 <= last:
        days = []
        for d in range(start, start + WINDOW_DAYS):
            v = _usable_value(series, d)
            if v is None:
                break
            days.append(v)
        if len(days) == WINDOW_DAYS:
            total = 0.0
            for v in days:
                total += v
            average = total / WINDOW_DAYS
            if best is None or average > best['average']:
                best = {'year': year, 'startDate': format_iso_date(start),
                        'endDate': format_iso_date(start + WINDOW_DAYS - 1), 'days': days, 'average': average}
        start += 1
    return best


MIN_MONTH_FRACTION = 0.5


def monthly_means(series):
    """Monthly mean for every month of every year the series touches (heatmaps)."""
    if not series['values']:
        return []
    first = from_day_number(series['startDay'])['year']
    last = from_day_number(series['startDay'] + len(series['values']) - 1)['year']
    cells = []
    for year in range(first, last + 1):
        for month in range(1, 13):
            dim = days_in_month(year, month)
            start = to_day_number(year, month, 1)
            total, days = 0.0, 0
            for d in range(start, start + dim):
                v = _usable_value(series, d)
                if v is None:
                    continue
                total += v
                days += 1
            cells.append({'year': year, 'month': month,
                          'mean': total / days if days >= MIN_MONTH_FRACTION * dim else None,
                          'days': days, 'daysInMonth': dim})
    return cells


def annual_min_for_year(series, year):
    best = None
    for d in range(to_day_number(year, 1, 1), to_day_number(year, 12, 31) + 1):
        v = _usable_value(series, d)
        if v is None:
            continue
        if best is None or v < best['value']:
            best = {'year': year, 'date': format_iso_date(d), 'value': v}
    return best


# ───────────────────────────── statistics and design ─────────────────────────────

RELIABILITY_PERCENT = 98
Z_98 = 2.0537489106318225
LOW_TEMP_DEPTH_MM = 0


def mean(values):
    total = 0.0
    for v in values:
        total += v
    return total / len(values)


def sample_standard_deviation(values):
    m = mean(values)
    squares = 0.0
    for v in values:
        squares += (v - m) ** 2
    return math.sqrt(squares / (len(values) - 1))


def high_temperature_design(mean7, sd7, latitude):
    lat = abs(latitude)
    air = mean7 + Z_98 * sd7
    return {'designAirTemp': air,
            'pavementTemp20mm': (air - 0.00618 * lat ** 2 + 0.2289 * lat + 42.2) * 0.9545 - 17.78}


def low_temperature_design(mean_min, sd_min, latitude):
    lat = abs(latitude)
    base = -1.56 + 0.72 * mean_min - 0.004 * lat ** 2 + 6.26 * math.log10(LOW_TEMP_DEPTH_MM + 25)
    term = Z_98 * math.sqrt(4.4 + 0.52 * sd_min ** 2)
    return {'meanAirTemp': mean_min, 'designAirTemp': mean_min, 'baseTemp': base,
            'reliabilityTerm': term, 'pavementTemp': base - term}


def _clean_t(t):
    return math.floor(t * 1e6 + 0.5) / 1e6  # same as JavaScript Math.round(t * 1e6) / 1e6


def round_high_grade(t):
    t = _clean_t(t)
    for g in HIGH_GRADES:
        if g >= t:
            return {'grade': g, 'outOfRange': False}
    return {'grade': HIGH_GRADES[-1], 'outOfRange': True}


def round_low_grade(t):
    t = _clean_t(t)
    for g in LOW_GRADES:
        if g <= t:
            return {'grade': g, 'outOfRange': False}
    return {'grade': LOW_GRADES[-1], 'outOfRange': True}


def speed_class(kmh):
    if kmh < SPEED_CLASS_LIMITS['standingBelowKmh']:
        return 'standing'
    if kmh <= SPEED_CLASS_LIMITS['slowUpToKmh']:
        return 'slow'
    return 'standard'


def esal_class(esal):
    found = ESAL_CLASSES[0][0]
    for cid, _label, frm in ESAL_CLASSES:
        if esal >= frm:
            found = cid
    return found


def adjust_for_traffic(climate_high, esal, speed, accept_consider=False):
    sc, ec = speed_class(speed), esal_class(esal)
    cell = GRADE_BUMP_TABLE[ec][sc]
    consider = cell == 'consider'
    added = (1 if accept_consider else 0) if consider else cell
    wanted = climate_high + added * GRADE_STEP_C
    return {'speedClass': sc, 'esalClass': ec, 'cell': cell, 'isConsiderCase': consider, 'gradesAdded': added,
            'highGradeBefore': climate_high, 'highGradeAfter': min(wanted, HIGH_GRADES[-1]),
            'cappedAtMax': wanted > HIGH_GRADES[-1]}


def specification_table(high, low):
    listed = M320_LOW_GRADES.get(high, [])
    lows = list(SPEC_TABLE_LOW_GRADES)
    if low not in lows:
        lows.append(low)
    columns = [{'lowGrade': lg, 'pavDsrTempC': (high + lg) / 2 + 4, 'bbrTempC': lg + 10, 'dttTempC': lg + 10,
                'selected': lg == low, 'inM320': lg in listed} for lg in lows]
    label = grade_label(high, low)
    return {'title': f'Specification for Bitumen with Performance Grade {label}', 'label': label,
            'highGrade': high, 'lowGrade': low, 'columns': columns,
            'selectedIndex': next(i for i, c in enumerate(columns) if c['selected']),
            'originalDsrTempC': high, 'rtfoDsrTempC': high, 'pavAgingTempLabel': pav_aging_temperature_label(high),
            'inM320': low in listed, 'limits': SPEC_LIMITS}


# Binder type for crude oil blends (user's chart, PG 46-76; PG 82/88 from the notebook). 1 = crude oil,
# 2 = high-quality crude oil, 3 = modifier required. Rows: low grade -10 ... -40.
BINDER_TYPES = {
    1: ('Crude oil', 'Standard unmodified (neat) binder from ordinary crude oil'),
    2: ('High-quality crude oil', 'Unmodified binder from selected, high-quality crude sources'),
    3: ('Modifier required', 'Polymer-modified binder (e.g. SBS) is needed'),
}
BINDER_MATRIX_HIGH = [46, 52, 58, 64, 70, 76, 82, 88]
BINDER_MATRIX_LOW = [-10, -16, -22, -28, -34, -40]
BINDER_MATRIX = {46: [1, 1, 1, 1, 1, 2], 52: [1, 1, 1, 1, 2, 3], 58: [1, 1, 1, 2, 2, 3], 64: [1, 1, 1, 2, 3, 3],
                 70: [1, 1, 2, 3, 3, 3], 76: [1, 2, 3, 3, 3, 3], 82: [2, 3, 3, 3, 3, 3], 88: [3, 3, 3, 3, 3, 3]}
RULE_OF_90 = 90


def binder_type(high, low):
    row = BINDER_MATRIX.get(high)
    if row is not None and low in BINDER_MATRIX_LOW:
        code, from_chart = row[BINDER_MATRIX_LOW.index(low)], True
    else:
        from_chart = False
        code = 3 if high >= 82 or low <= -34 else 2 if high >= 70 or low <= -28 else 1
    total = high + abs(low)
    return {'code': code, 'name': BINDER_TYPES[code][0], 'description': BINDER_TYPES[code][1], 'fromChart': from_chart,
            'temperatureSum': total, 'ruleOf90SaysModified': total > RULE_OF_90}


SPEC_TABLE_NOTE = ('The highlighted column (orange background) indicates the test temperatures corresponding to the '
                   'project selected PG grade. All values must meet the specified criteria for the binder to be '
                   'certified for this performance grade according to AASHTO M320/M332 standards.')
DTT_FOOTNOTE = ('Under AASHTO M320, the direct tension test is needed only when the BBR creep stiffness S is '
                'between 300 and 600 MPa (with m ≥ 0.300).')

INPUT_LIMITS = {'latitude': (-90, 90), 'designEsalMillions': (0, 1000), 'designSpeedKmh': (0, 200)}


def validate_design_inputs(inputs):
    problems = []
    for field, name, unit in (('latitude', 'Latitude', '°'), ('designEsalMillions', 'Design ESAL', ' million'),
                              ('designSpeedKmh', 'Design speed', ' km/h')):
        v = inputs.get(field)
        lo, hi = INPUT_LIMITS[field]
        if not _finite(v) or v < lo or v > hi:
            problems.append(f'{name} must be a number from {lo} to {hi}{unit}.')
    return problems


MIN_REAL_FRACTION = 0.75
MIN_YEARS = 5


def _collect_years(series, pick):
    """Used years, plus every left-out year with its share of recorded days (shown greyed out)."""
    years, excluded_years, excluded = [], [], []
    for c in year_coverage(series):
        enough = c['realFraction'] >= MIN_REAL_FRACTION
        result = pick(series, c['year']) if enough else None
        if result is not None:
            years.append(result)
            continue
        excluded_years.append(c['year'])
        excluded.append({'year': c['year'], 'realFraction': c['realFraction'],
                         'reason': 'no-usable-period' if enough else 'too-few-days'})
    return {'years': years, 'excludedYears': excluded_years, 'excluded': excluded}


def _too_few(found, name):
    what = 'No year has' if found == 0 else 'Only 1 year has' if found == 1 else f'Only {found} years have'
    return (f'{what} enough {name} data. At least {MIN_YEARS} years with '
            f'{round(MIN_REAL_FRACTION * 100)}% of days recorded are needed.')


def analyse_climate(observations):
    series, quality = clean_observations(observations)
    if not series['tmax']['values'] and not series['tmin']['values']:
        raise AnalysisError('NO_DATA', 'The file has no temperature values. Check the columns and try again.')
    high = _collect_years(series['tmax'], seven_day_max_for_year)
    low = _collect_years(series['tmin'], annual_min_for_year)
    if len(high['years']) < MIN_YEARS:
        raise AnalysisError('TOO_FEW_YEARS_HIGH', _too_few(len(high['years']), 'maximum temperature (Tmax)'))
    if len(low['years']) < MIN_YEARS:
        raise AnalysisError('TOO_FEW_YEARS_LOW', _too_few(len(low['years']), 'minimum temperature (Tmin)'))
    monthly = {'tmax': monthly_means(series['tmax']), 'tmin': monthly_means(series['tmin'])}
    return {'engineVersion': ENGINE_VERSION, 'high': high, 'low': low, 'monthly': monthly, 'quality': quality}


def compute_grade(climate, inputs):
    problems = validate_design_inputs(inputs)
    if problems:
        raise AnalysisError('INVALID_INPUT', problems[0])
    hv = [y['average'] for y in climate['high']['years']]
    h_mean, h_sd = mean(hv), sample_standard_deviation(hv)
    high = high_temperature_design(h_mean, h_sd, inputs['latitude'])
    hg = round_high_grade(high['pavementTemp20mm'])
    hottest = climate['high']['years'][0]
    for y in climate['high']['years']:
        if y['average'] > hottest['average']:
            hottest = y
    lv = [y['value'] for y in climate['low']['years']]
    l_mean, l_sd = mean(lv), sample_standard_deviation(lv)
    low = low_temperature_design(l_mean, l_sd, inputs['latitude'])
    lg = round_low_grade(low['pavementTemp'])
    coldest = climate['low']['years'][0]
    for y in climate['low']['years']:
        if y['value'] < coldest['value']:
            coldest = y
    traffic = adjust_for_traffic(hg['grade'], inputs['designEsalMillions'], inputs['designSpeedKmh'],
                                 bool(inputs.get('acceptConsiderBump', False)))
    return {
        'engineVersion': ENGINE_VERSION, 'reliabilityPercent': RELIABILITY_PERCENT, 'z': Z_98,
        'high': {'yearsUsed': len(hv), 'mean': h_mean, 'sd': h_sd, 'designAirTemp': high['designAirTemp'],
                 'pavementTemp': high['pavementTemp20mm'], 'grade': hg['grade'], 'outOfRange': hg['outOfRange'],
                 'hottestYear': hottest['year']},
        'low': {'yearsUsed': len(lv), 'mean': l_mean, 'sd': l_sd, 'meanAirTemp': low['meanAirTemp'],
                'designAirTemp': low['designAirTemp'], 'baseTemp': low['baseTemp'],
                'reliabilityTerm': low['reliabilityTerm'], 'pavementTemp': low['pavementTemp'],
                'grade': lg['grade'], 'outOfRange': lg['outOfRange'], 'coldestYear': coldest['year']},
        'climateGrade': {'high': hg['grade'], 'low': lg['grade'], 'label': grade_label(hg['grade'], lg['grade'])},
        'traffic': traffic,
        'finalGrade': {'high': traffic['highGradeAfter'], 'low': lg['grade'],
                       'label': grade_label(traffic['highGradeAfter'], lg['grade'])},
        'binderType': binder_type(traffic['highGradeAfter'], lg['grade']),
        'spec': specification_table(traffic['highGradeAfter'], lg['grade']),
    }


def analyse(observations, inputs):
    climate = analyse_climate(observations)
    return climate, compute_grade(climate, inputs)


# ═════════════════════════════ file reading (parser) ═════════════════════════════


def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)


def is_blank(cell):
    if cell is None:
        return True
    if isinstance(cell, float) and math.isnan(cell):
        return True
    return isinstance(cell, str) and cell.strip() == ''


def _js_number_text(v):
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


def cell_text(cell):
    if isinstance(cell, str):
        return cell.strip()
    if isinstance(cell, bool):
        return 'true' if cell else 'false'
    if _num(cell) and math.isfinite(cell):
        return _js_number_text(cell)
    return ''


_PARENS = re.compile(r'\(.*?\)|\[.*?\]')
_UNITS = re.compile(r'°\s*[cf]\b|\bdeg(?:rees?)?\s*[cf]\b|\bcelsius\b|\bfahrenheit\b', re.ASCII)
_COMBINING = re.compile('[̀-ͯ]')


def text_key(text):
    s = unicodedata.normalize('NFKD', text.lower())
    s = _COMBINING.sub('', s)
    s = _PARENS.sub(' ', s)
    s = _UNITS.sub(' ', s)
    return re.sub(r'[^a-z0-9]', '', s)


_FAHRENHEIT = re.compile(r'°\s*f\b|\(\s*f\s*\)|\[\s*f\s*\]|\bdeg(?:rees?)?\s*f\b|fahrenheit', re.I | re.ASCII)


def mentions_fahrenheit(text):
    return bool(_FAHRENHEIT.search(text))


_NUMBER = re.compile(r'^([+-]?(?:[0-9]+(?:[.,][0-9]*)?|[.,][0-9]+))\s*(?:°\s*[cf]?|[*#a-z]{0,2})$', re.I)


def to_number(cell):
    if _num(cell):
        return float(cell) if math.isfinite(cell) else None
    if not isinstance(cell, str):
        return None
    s = cell.strip().replace('−', '-').replace('–', '-')
    m = _NUMBER.match(s)
    if not m:
        return None
    n = float(m.group(1).replace(',', '.'))
    return n if math.isfinite(n) else None


def to_integer(cell):
    n = to_number(cell)
    return int(n) if n is not None and float(n).is_integer() else None


ROLE_NAMES = {
    'station': ['name', 'station', 'stationname', 'stn', 'stnname', 'site', 'sitename', 'stationid', 'location'],
    'latitude': ['lat', 'latitude', 'lati', 'latdeg', 'geogr2'],
    'longitude': ['lon', 'long', 'longitude', 'lng', 'longdeg', 'geogr1'],
    'elevation': ['elevation', 'elev', 'alt', 'altitude', 'height', 'elevationm'],
    'element': ['element', 'elem', 'elementcode', 'elementname', 'variable', 'var', 'parameter', 'param',
                'measurement', 'type'],
    'year': ['year', 'yr', 'yyyy', 'years', 'yy'],
    'month': ['month', 'mon', 'mm', 'mo', 'months', 'mth'],
    'day': ['day', 'dd', 'dy', 'dayofmonth'],
    'date': ['date', 'dates', 'datetime', 'obsdate', 'observationdate', 'timestamp', 'dateyyyymmdd', 'ddmmyyyy',
             'yyyymmdd', 'dailydate'],
    'value': ['value', 'values', 'val', 'temperature', 'temp', 'reading', 'obs', 'observation', 'data', 'amount'],
}
ELEMENT_NAMES = {
    'tmax': ['tmax', 'tmpmax', 'tempmax', 'maxtemp', 'maxtemperature', 'maximumtemperature', 'maximumtemp', 'maxtmp',
             'tx', 'max', 'maximum', 'tmaximum', 'airtempmax', 'temperaturemax', 'high', 'hightemp', 'dailymax',
             'dailymaximum', 'maxairtemperature', 'maximumairtemperature', 'tmx'],
    'tmin': ['tmin', 'tmpmin', 'tempmin', 'mintemp', 'mintemperature', 'minimumtemperature', 'minimumtemp', 'mintmp',
             'tn', 'min', 'minimum', 'tminimum', 'airtempmin', 'temperaturemin', 'low', 'lowtemp', 'dailymin',
             'dailyminimum', 'minairtemperature', 'minimumairtemperature', 'tmn'],
}
MONTH_NAMES = [['jan', 'january'], ['feb', 'february'], ['mar', 'march'], ['apr', 'april'], ['may'],
               ['jun', 'june'], ['jul', 'july'], ['aug', 'august'], ['sep', 'sept', 'september'],
               ['oct', 'october'], ['nov', 'november'], ['dec', 'december']]


def month_from_name(text):
    key = text_key(text)
    for i, names in enumerate(MONTH_NAMES):
        if key in names:
            return i + 1
    return None


def element_from_text(text):
    key = text_key(text)
    if not key:
        return None
    if key in ELEMENT_NAMES['tmax']:
        return 'tmax'
    if key in ELEMENT_NAMES['tmin']:
        return 'tmin'
    if 'minute' in key:
        return None
    temp_word = 'temp' in key or 'tmp' in key or key.startswith('t')
    is_max, is_min = 'max' in key, 'min' in key
    if temp_word and is_max and not is_min:
        return 'tmax'
    if temp_word and is_min and not is_max:
        return 'tmin'
    return None


def element_from_name(text):
    whole = element_from_text(text)
    if whole:
        return whole
    for word in re.split(r'[^A-Za-z0-9]+', text):
        e = element_from_text(word)
        if e:
            return e
    return None


_DAY_HEADER = re.compile(r'^(?:day|d)?0?([1-9]|[12][0-9]|3[01])(?:st|nd|rd|th)?$')


def classify_header(cell):
    if _num(cell):
        if math.isfinite(cell) and float(cell).is_integer() and 1 <= cell <= 31:
            return {'dayOfMonth': int(cell), 'fahrenheit': False}
        return {'fahrenheit': False}
    text = cell_text(cell)
    fahrenheit = mentions_fahrenheit(text)
    key = text_key(text)
    if not key:
        return {'fahrenheit': fahrenheit}
    for role, names in ROLE_NAMES.items():
        if key in names:
            return {'role': role, 'fahrenheit': fahrenheit}
    m = _DAY_HEADER.match(key)
    if m:
        return {'dayOfMonth': int(m.group(1)), 'fahrenheit': fahrenheit}
    month = month_from_name(text)
    if month is not None:
        return {'monthOfYear': month, 'fahrenheit': fahrenheit}
    el = element_from_text(text)
    if el:
        return {'role': el, 'fahrenheit': fahrenheit}
    return {'fahrenheit': fahrenheit}


# ── dates in cells ──

def excel_serial_to_date(serial, date1904=False):
    s = math.floor(serial)
    if date1904:
        s += 1462
    if s < 1 or s > 2958465:
        return None
    if s == 60:
        return {'year': 1900, 'month': 2, 'day': 29}
    return from_day_number(s - 25569 if s > 60 else s - 25568)


_TIME_SUFFIX = re.compile(r'(?:[T\s]+[0-9]{1,2}:[0-9]{2}(?::[0-9]{2}(?:\.[0-9]+)?)?\s*(?:am|pm)?)$', re.I)
_ISO = re.compile(r'^([0-9]{4})[-/.]([0-9]{1,2})[-/.]([0-9]{1,2})$')
_COMPACT = re.compile(r'^([0-9]{4})([0-9]{2})([0-9]{2})$')
_NUMERIC_DMY = re.compile(r'^([0-9]{1,2})[-/.]([0-9]{1,2})[-/.]([0-9]{2}|[0-9]{4})$')


def _full_year_text(t):
    y = int(t)
    if len(t) == 4:
        return y
    return 2000 + y if y < 50 else 1900 + y


def _from_compact_number(n):
    if not float(n).is_integer() or n < 18000101 or n > 22001231:
        return None
    n = int(n)
    return {'year': n // 10000, 'month': (n // 100) % 100, 'day': n % 100}


def detect_date_order(cells):
    day_first = month_first = False
    for cell in cells:
        if not isinstance(cell, str):
            continue
        m = _NUMERIC_DMY.match(_TIME_SUFFIX.sub('', cell.strip()))
        if not m:
            continue
        if int(m.group(1)) > 12:
            day_first = True
        if int(m.group(2)) > 12:
            month_first = True
    return 'mdy' if month_first and not day_first else 'dmy'


def parse_date_cell(cell, order='dmy', date1904=False):
    if isinstance(cell, (datetime, date)):
        return {'year': cell.year, 'month': cell.month, 'day': cell.day}
    if _num(cell):
        if not math.isfinite(cell):
            return None
        return _from_compact_number(cell) or excel_serial_to_date(cell, date1904)
    if not isinstance(cell, str):
        return None
    s = _TIME_SUFFIX.sub('', cell.strip()).strip()
    m = _ISO.match(s)
    if m:
        return {'year': int(m.group(1)), 'month': int(m.group(2)), 'day': int(m.group(3))}
    m = _COMPACT.match(s)
    if m:
        return {'year': int(m.group(1)), 'month': int(m.group(2)), 'day': int(m.group(3))}
    m = _NUMERIC_DMY.match(s)
    if m:
        a, b, y = int(m.group(1)), int(m.group(2)), _full_year_text(m.group(3))
        return {'year': y, 'month': a, 'day': b} if order == 'mdy' else {'year': y, 'month': b, 'day': a}
    return _parse_word_date(s)


def _parse_word_date(s):
    tokens = [t for t in re.split(r'[\s,\-/.]+', s) if t]
    if len(tokens) != 3:
        return None
    month = year = day = None
    for t in tokens:
        name = month_from_name(t) if re.fullmatch(r'[a-z]+', t, re.I) else None
        if name is not None and month is None:
            month = name
        elif re.fullmatch(r'[0-9]{4}', t) and year is None:
            year = int(t)
        elif re.fullmatch(r'[0-9]{1,2}', t) and day is None:
            day = int(t)
        else:
            return None
    return {'year': year, 'month': month, 'day': day} if None not in (month, year, day) else None


def parse_year_month_cell(cell, date1904=False):
    if isinstance(cell, str):
        s = cell.strip()
        m = re.match(r'^([0-9]{4})[-/.]([0-9]{1,2})$', s)
        if m:
            return {'year': int(m.group(1)), 'month': int(m.group(2))}
        m = re.match(r'^([0-9]{1,2})[-/.]([0-9]{4})$', s)
        if m:
            return {'year': int(m.group(2)), 'month': int(m.group(1))}
        tokens = [t for t in re.split(r'[\s,\-/.]+', s) if t]
        if len(tokens) == 2:
            a, b = tokens
            month = month_from_name(b if re.match(r'^[0-9]', a) else a)
            year = int(a) if re.fullmatch(r'[0-9]{4}', a) else int(b) if re.fullmatch(r'[0-9]{4}', b) else None
            if month is not None and year is not None:
                return {'year': year, 'month': month}
    d = parse_date_cell(cell, 'dmy', date1904)
    return {'year': d['year'], 'month': d['month']} if d else None


def looks_like_date(cell):
    if isinstance(cell, (datetime, date)):
        return True
    if _num(cell):
        return math.isfinite(cell) and _from_compact_number(cell) is not None
    if not isinstance(cell, str):
        return False
    s = _TIME_SUFFIX.sub('', cell.strip()).strip()
    return bool(_ISO.match(s) or _COMPACT.match(s) or _NUMERIC_DMY.match(s)) or _parse_word_date(s) is not None


# ── layouts ──

def _at(row, idx):
    if row is None or idx is None or idx < 0 or idx >= len(row):
        return None
    return row[idx]


def _map_columns(row):
    m = {'roles': {}, 'days': {}, 'months': {}, 'fahrenheit': set()}
    for col, cell in enumerate(row):
        info = classify_header(cell)
        if info.get('fahrenheit'):
            m['fahrenheit'].add(col)
        role = info.get('role')
        if role and role not in m['roles']:
            m['roles'][role] = col
        d = info.get('dayOfMonth')
        if d and d not in m['days']:
            m['days'][d] = col
        mo = info.get('monthOfYear')
        if mo and mo not in m['months']:
            m['months'][mo] = col
    return m


def _has_date(m):
    r = m['roles']
    return 'date' in r or ('year' in r and 'month' in r and 'day' in r)


def _layout_of(m):
    r = m['roles']
    if len(m['days']) >= 28 and (('year' in r and 'month' in r) or 'date' in r or 'month' in r):
        return 'wide-days'
    if len(m['months']) == 12 and 'year' in r and 'day' in r:
        return 'wide-months'
    if _has_date(m) and ('tmax' in r or 'tmin' in r):
        return 'long'
    if _has_date(m) and 'element' in r and 'value' in r:
        return 'long-element'
    return None


def _score(m):
    return len(m['roles']) + len(m['days']) + len(m['months'])


def _find_blocks(rows):
    found = []
    for i, row in enumerate(rows[:200_000]):
        if not row or all(is_blank(c) for c in row):
            continue
        m = _map_columns(row)
        layout = _layout_of(m)
        if layout:
            found.append({'row': i, 'map': m, 'layout': layout})
    if not found:
        return []
    best = found[0]
    for f in found[1:]:
        if _score(f['map']) > _score(best['map']):
            best = f
    same = [f for f in found if f['layout'] == best['layout']]
    return [{'headerRow': f['row'], 'endRow': same[i + 1]['row'] if i + 1 < len(same) else len(rows),
             'map': f['map'], 'layout': f['layout']} for i, f in enumerate(same)]


def _title_text_above(rows, header_row, stop_row):
    parts = []
    for i in range(header_row - 1, max(stop_row, header_row - 5) - 1, -1):
        for cell in rows[i] or []:
            if isinstance(cell, str) and cell.strip():
                parts.append(cell.strip())
    return ' '.join(parts)


def _element_from_title(text):
    for part in re.split(r'\s{2,}|[|;]', text):
        e = element_from_text(part)
        if e:
            return e
    return element_from_text(text)


def _valid_coordinate(n, mx):
    return n if n is not None and abs(n) <= mx else None


def _metadata_above(rows, header_row, station):
    for i in range(header_row):
        row = rows[i] or []
        for c, cell in enumerate(row):
            text = cell_text(cell)
            if not text:
                continue
            label, value = text, _at(row, c + 1)
            colon = text.find(':')
            if colon > 0:
                label = text[:colon]
                rest = text[colon + 1:].strip()
                if rest:
                    value = rest
            role = classify_header(label).get('role')
            if role == 'station' and station['name'] is None and cell_text(value):
                station['name'] = cell_text(value)
            if role == 'latitude' and station['latitude'] is None:
                station['latitude'] = _valid_coordinate(to_number(value), 90)
            if role == 'longitude' and station['longitude'] is None:
                station['longitude'] = _valid_coordinate(to_number(value), 180)
            if role == 'elevation' and station['elevation'] is None:
                station['elevation'] = to_number(value)


def _full_year(n):
    if n is None:
        return None
    if 0 <= n < 100:
        return 2000 + n if n < 50 else 1900 + n
    return n


def _read_station_columns(row, m, out):
    r = m['roles']
    if 'station' in r:
        name = cell_text(_at(row, r['station']))
        if name:
            out['stationNames'].setdefault(name, None)
    st = out['station']
    if 'latitude' in r and st['latitude'] is None:
        st['latitude'] = _valid_coordinate(to_number(_at(row, r['latitude'])), 90)
    if 'longitude' in r and st['longitude'] is None:
        st['longitude'] = _valid_coordinate(to_number(_at(row, r['longitude'])), 180)
    if 'elevation' in r and st['elevation'] is None:
        st['elevation'] = to_number(_at(row, r['elevation']))


def _row_element(row, m, fallback, out):
    if 'element' not in m['roles']:
        return fallback
    text = cell_text(_at(row, m['roles']['element']))
    el = element_from_text(text)
    if not el and text:
        out['unknownElements'].setdefault(text, None)
    return el


def _row_date(row, m, order, date1904):
    r = m['roles']
    if 'date' in r:
        d = parse_date_cell(_at(row, r['date']), order, date1904)
        if d:
            return d
    if 'year' in r and 'month' in r and 'day' in r:
        y = _full_year(to_integer(_at(row, r['year'])))
        mo = to_integer(_at(row, r['month']))
        d = to_integer(_at(row, r['day']))
        if None not in (y, mo, d):
            return {'year': y, 'month': mo, 'day': d}
    return None


def _extract_block(rows, block, context, date1904, out):
    m, r, layout = block['map'], block['map']['roles'], block['layout']
    data_rows = rows[block['headerRow'] + 1:block['endRow']]
    order = detect_date_order([_at(row, r['date']) for row in data_rows]) if 'date' in r else 'dmy'
    if 'date' in r and layout != 'wide-days':
        out['dateOrder'] = order

    def convert(v, col):
        if v is None or not (col in m['fahrenheit'] or context['fahrenheit']):
            return v
        out['fahrenheit'] = True
        return ((v - 32) * 5) / 9

    for row in data_rows:
        if not row or all(is_blank(c) for c in row):
            continue
        used = False
        if layout == 'wide-days':
            year = _full_year(to_integer(_at(row, r['year']))) if 'year' in r else None
            month = to_integer(_at(row, r['month'])) if 'month' in r else None
            if year is None or month is None:
                ym_col = r['date'] if 'date' in r else r.get('month')
                ym = parse_year_month_cell(_at(row, ym_col), date1904)
                if ym:
                    year, month = ym['year'], ym['month']
            el = _row_element(row, m, context['element'], out)
            if year is not None and month is not None and 1 <= month <= 12 and el:
                for day, col in m['days'].items():
                    raw = _at(row, col)
                    if is_blank(raw):
                        continue
                    out['observations'].append({'year': year, 'month': month, 'day': day,
                                                el: convert(to_number(raw), col)})
                used = True
        elif layout == 'wide-months':
            year = _full_year(to_integer(_at(row, r['year'])))
            day = to_integer(_at(row, r['day']))
            el = _row_element(row, m, context['element'], out)
            if year is not None and day is not None and 1 <= day <= 31 and el:
                for month, col in m['months'].items():
                    raw = _at(row, col)
                    if is_blank(raw):
                        continue
                    out['observations'].append({'year': year, 'month': month, 'day': day,
                                                el: convert(to_number(raw), col)})
                used = True
        elif layout == 'long':
            d = _row_date(row, m, order, date1904)
            if d:
                obs = dict(d)
                if 'tmax' in r:
                    obs['tmax'] = convert(to_number(_at(row, r['tmax'])), r['tmax'])
                if 'tmin' in r:
                    obs['tmin'] = convert(to_number(_at(row, r['tmin'])), r['tmin'])
                out['observations'].append(obs)
                used = True
        elif layout == 'long-element':
            d = _row_date(row, m, order, date1904)
            el = _row_element(row, m, None, out)
            if d and el:
                obs = dict(d)
                obs[el] = convert(to_number(_at(row, r['value'])), r['value'])
                out['observations'].append(obs)
                used = True
        if used:
            out['rowsUsed'] += 1
            _read_station_columns(row, m, out)
        else:
            out['rowsSkipped'] += 1


def _extract_by_content(sheet, date1904, out):
    rows = sheet['rows']
    width = max((len(r) for r in rows if r), default=0)
    columns = [[_at(r, c) for r in rows] for c in range(width)]

    def non_blank(cells):
        return [c for c in cells if not is_blank(c)]

    date_col = None
    for c, cells in enumerate(columns):
        filled = non_blank(cells)
        if len(filled) >= 10 and sum(1 for x in filled if looks_like_date(x)) >= 0.8 * len(filled):
            date_col = c
            break
    if date_col is None:
        return False
    temps = []
    for c, cells in enumerate(columns):
        if c == date_col:
            continue
        filled = non_blank(cells)
        numbers = [n for n in (to_number(x) for x in filled) if n is not None]
        if len(numbers) < 10 or len(numbers) < 0.8 * len(filled):
            continue
        if sum(1 for n in numbers if -50 <= n <= 60) < 0.9 * len(numbers):
            continue
        if all(float(n).is_integer() for n in numbers) and all(1 <= n <= 31 for n in numbers):
            continue
        total = 0.0
        for n in numbers:
            total += n
        temps.append({'c': c, 'mean': total / len(numbers)})
    if len(temps) != 2:
        raise file_error(
            'The file has dates but the Tmax and Tmin columns could not be found. Name the columns Tmax and Tmin.'
            if len(temps) < 2 else
            'The file has more than two number columns. Name the temperature columns Tmax and Tmin.')
    tmax, tmin = (temps[0], temps[1]) if temps[0]['mean'] >= temps[1]['mean'] else (temps[1], temps[0])
    order = detect_date_order(columns[date_col])
    out['dateOrder'] = order
    for row in rows:
        d = parse_date_cell(_at(row, date_col), order, date1904) if row else None
        if not d:
            if row and not all(is_blank(c) for c in row):
                out['rowsSkipped'] += 1
            continue
        out['observations'].append({**d, 'tmax': to_number(_at(row, tmax['c'])),
                                    'tmin': to_number(_at(row, tmin['c']))})
        out['rowsUsed'] += 1
    return True


def extract_dataset(sheets, file_name='', date1904=False):
    out = {'observations': [], 'stationNames': {}, 'rowsUsed': 0, 'rowsSkipped': 0, 'fahrenheit': False,
           'dateOrder': None, 'unknownElements': {},
           'station': {'name': None, 'latitude': None, 'longitude': None, 'elevation': None}}
    sheets_used, layout, missing_element = [], None, False
    for sheet in sheets:
        rows = sheet['rows']
        blocks = _find_blocks(rows)
        if not blocks:
            continue
        _metadata_above(rows, blocks[0]['headerRow'], out['station'])
        before = len(out['observations'])
        for i, block in enumerate(blocks):
            title = _title_text_above(rows, block['headerRow'], 0 if i == 0 else blocks[i - 1]['headerRow'] + 1)
            needs = block['layout'] in ('wide-days', 'wide-months') and 'element' not in block['map']['roles']
            el = (_element_from_title(title) or element_from_name(sheet['name'])
                  or element_from_name(re.sub(r'\.[^.]+$', '', file_name))) if needs else None
            if needs and not el:
                missing_element = True
            _extract_block(rows, block, {'element': el, 'fahrenheit': mentions_fahrenheit(title)}, date1904, out)
        if len(out['observations']) > before:
            sheets_used.append(sheet['name'])
            layout = layout or blocks[0]['layout']

    if not out['observations'] and not missing_element and not out['unknownElements']:
        for sheet in sheets:
            if _extract_by_content(sheet, date1904, out):
                sheets_used.append(sheet['name'])
                layout = 'auto-columns'
                break

    if not out['observations']:
        if missing_element:
            raise file_error('Could not tell whether the values are maximum or minimum temperatures. Add an element '
                             'column (TMPMAX / TMPMIN) or name the sheet Tmax / Tmin.')
        if out['unknownElements']:
            found = ', '.join(list(out['unknownElements'])[:5])
            raise file_error(f'No TMPMAX or TMPMIN rows were found in the element column (found: {found}).')
        raise file_error('The temperature data could not be found in this file. It needs dates (or Year and Month) '
                         'with Tmax and Tmin values. Download the template to see an example.')
    names = list(out['stationNames'])
    if len(names) > 1:
        raise file_error(f'The file has more than one station ({", ".join(names[:3])}). Upload one station per file.')
    if out['station']['name'] is None and names:
        out['station']['name'] = names[0]

    def has_value(el):
        return any(o.get(el) is not None for o in out['observations'])

    if not has_value('tmax') and not has_value('tmin'):
        raise file_error('The file has no temperature values.')
    if not has_value('tmax'):
        raise file_error('Only minimum temperatures (Tmin) were found. The file also needs maximum temperatures (Tmax).')
    if not has_value('tmin'):
        raise file_error('Only maximum temperatures (Tmax) were found. The file also needs minimum temperatures (Tmin).')
    return {'layout': layout or 'auto-columns', 'station': out['station'], 'observations': out['observations'],
            'details': {'sheetsUsed': sheets_used, 'rowsUsed': out['rowsUsed'], 'rowsSkipped': out['rowsSkipped'],
                        'convertedFromFahrenheit': out['fahrenheit'], 'dateOrder': out['dateOrder']}}


# ── reading the file bytes ──

MAX_FILE_BYTES = 10 * 1024 * 1024
_MAX_ROWS, _MAX_COLS = 200_000, 200


def _decode_text(data):
    try:
        text = data.decode('utf-8')
    except UnicodeDecodeError:
        text = data.decode('cp1252', errors='replace')
    return text[1:] if text.startswith('﻿') else text


def _guess_delimiter(text):
    lines = [ln for ln in text.splitlines() if ln.strip()][:20]
    best, best_score = ',', -1.0
    for d in (',', '\t', '|', ';'):
        counts = [len(r) for r in csv.reader(lines, delimiter=d)]
        if not counts:
            continue
        mode = max(set(counts), key=counts.count)
        if mode < 2:
            continue
        score = mode * counts.count(mode) / len(counts)
        if score > best_score:
            best, best_score = d, score
    return best


def _read_csv(data, file_name):
    text = _decode_text(data)
    if '\x00' in text:
        raise file_error('The file could not be read. Upload a CSV or Excel file.')
    rows = [r[:_MAX_COLS] for r in csv.reader(io.StringIO(text), delimiter=_guess_delimiter(text))]
    return [{'name': file_name, 'rows': rows[:_MAX_ROWS]}]


def _clean_excel_value(v):
    if isinstance(v, float) and math.isnan(v):
        return None
    if hasattr(v, 'to_pydatetime'):
        return v.to_pydatetime()
    return v


def _read_workbook(data, file_name):
    if data[:4] == b'PK\x03\x04':
        try:
            import openpyxl
            wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True, data_only=True)
            sheets = []
            for ws in wb.worksheets:
                rows = []
                for row in ws.iter_rows(values_only=True, max_row=_MAX_ROWS):
                    rows.append([_clean_excel_value(v) for v in row[:_MAX_COLS]])
                sheets.append({'name': ws.title, 'rows': rows})
            return sheets
        except Exception:
            pass  # maybe an .ods file: try pandas below
    try:
        import pandas as pd
        frames = pd.read_excel(io.BytesIO(data), sheet_name=None, header=None)
    except Exception:
        raise file_error('The file could not be read. Upload a CSV or Excel file.')
    return [{'name': str(name), 'rows': [[_clean_excel_value(v) for v in r[:_MAX_COLS]]
                                         for r in df.itertuples(index=False, name=None)][:_MAX_ROWS]}
            for name, df in frames.items()]


def read_temperature_file(data, file_name):
    if len(data) == 0:
        raise file_error('The file is empty.')
    if len(data) > MAX_FILE_BYTES:
        raise file_error('The file is larger than 10 MB. Upload one station at a time.')
    if data[:4] == b'PK\x03\x04' or data[:4] == b'\xd0\xcf\x11\xe0':
        return extract_dataset(_read_workbook(data, file_name), file_name)
    return extract_dataset(_read_csv(data, file_name), file_name)


LAB_EQUIPMENT = [{'id': 'flash', 'name': 'Flash Point', 'device': 'Cleveland open cup', 'standard': 'AASHTO T 48', 'stage': 'Original binder', 'color': '#8e44ad', 'purpose': 'Safety check. The binder is heated in an open cup while a small flame passes over it; the flash point is the temperature at which the vapours first ignite briefly.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='18' y='98' width='124' height='12' rx='3' fill='#e5e7eb' stroke='#1f2937' stroke-width='2'/><rect x='44' y='86' width='72' height='12' fill='#9ca3af' stroke='#1f2937' stroke-width='2'/><path d='M54 86 L58 62 L102 62 L106 86 Z' fill='#f4d47c' stroke='#1f2937' stroke-width='2'/><rect x='60' y='70' width='40' height='10' fill='#111827'/><rect x='97' y='12' width='7' height='58' rx='3.5' fill='#ffffff' stroke='#1f2937' stroke-width='2'/><line x1='100.5' y1='30' x2='100.5' y2='68' stroke='#e34948' stroke-width='3'/><circle cx='100.5' cy='70' r='5' fill='#e34948' stroke='#1f2937' stroke-width='2'/><line x1='14' y1='46' x2='66' y2='54' stroke='#1f2937' stroke-width='3' stroke-linecap='round'/><path d='M70 54 C76 46 74 40 69 34 C68 42 62 44 65 54 Z' fill='#f59e0b' stroke='#b45309' stroke-width='1.5'/></svg>"}, {'id': 'rv', 'name': 'Rotational Viscometer', 'device': 'Brookfield-type viscometer with heated chamber', 'standard': 'AASHTO T 316', 'stage': 'Original binder', 'color': '#8e44ad', 'purpose': 'A spindle turns in the hot binder and the torque gives its viscosity at 135 °C. It shows that the binder can be pumped and mixed, and gives mixing and compaction temperatures.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='26' y='100' width='108' height='10' rx='3' fill='#e5e7eb' stroke='#1f2937' stroke-width='2'/><rect x='36' y='14' width='8' height='86' fill='#9ca3af' stroke='#1f2937' stroke-width='2'/><rect x='44' y='18' width='70' height='30' rx='5' fill='#2b8bd8' stroke='#1f2937' stroke-width='2'/><rect x='52' y='24' width='30' height='16' rx='2' fill='#d1fae5' stroke='#1f2937' stroke-width='1.5'/><circle cx='100' cy='32' r='5' fill='#ffffff' stroke='#1f2937' stroke-width='1.5'/><line x1='88' y1='48' x2='88' y2='86' stroke='#1f2937' stroke-width='3'/><rect x='82' y='76' width='12' height='14' rx='2' fill='#6b7280' stroke='#1f2937' stroke-width='1.5'/><rect x='68' y='64' width='40' height='36' rx='3' fill='#fca5a5' fill-opacity='0.6' stroke='#1f2937' stroke-width='2'/><rect x='72' y='78' width='32' height='18' fill='#111827' fill-opacity='0.85'/><path d='M76 58 A14 5 0 0 0 100 58' fill='none' stroke='#e34948' stroke-width='2' marker-end=''/><path d='M100 58 l-4 -3 m4 3 l-4 3' stroke='#e34948' stroke-width='2' fill='none'/></svg>"}, {'id': 'dsr', 'name': 'Dynamic Shear Rheometer', 'device': 'Parallel-plate rheometer', 'standard': 'AASHTO T 315', 'stage': 'Original, RTFO and PAV binder', 'color': '#17a589', 'purpose': 'A thin binder disc between two plates is twisted back and forth. The complex modulus G* and phase angle δ describe stiffness and elasticity: G*/sin δ controls rutting, G*·sin δ controls fatigue cracking.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='22' y='100' width='116' height='10' rx='3' fill='#e5e7eb' stroke='#1f2937' stroke-width='2'/><rect x='50' y='78' width='60' height='22' fill='#9ca3af' stroke='#1f2937' stroke-width='2'/><rect x='56' y='66' width='48' height='12' rx='2' fill='#d1d5db' stroke='#1f2937' stroke-width='2'/><rect x='58' y='60' width='44' height='6' fill='#111827'/><rect x='56' y='48' width='48' height='12' rx='2' fill='#d1d5db' stroke='#1f2937' stroke-width='2'/><rect x='75' y='10' width='10' height='38' fill='#6b7280' stroke='#1f2937' stroke-width='2'/><path d='M50 30 A30 9 0 0 0 110 30' fill='none' stroke='#17a589' stroke-width='2.5'/><path d='M110 30 l-6 -3 m6 3 l-5 4' stroke='#17a589' stroke-width='2.5' fill='none'/><path d='M50 30 l6 -3 m-6 3 l5 4' stroke='#17a589' stroke-width='2.5' fill='none'/><text x='118' y='66' font-size='10' font-family='Arial' fill='#1f2937'>sample</text><line x1='104' y1='63' x2='116' y2='63' stroke='#1f2937' stroke-width='1'/></svg>"}, {'id': 'rtfo', 'name': 'Rolling Thin Film Oven', 'device': 'RTFO with rotating bottle carriage', 'standard': 'AASHTO T 240', 'stage': 'Short-term ageing', 'color': '#e74c3c', 'purpose': 'Simulates ageing during mixing and laying: thin binder films in rotating glass bottles are exposed to hot air at 163 °C for 85 minutes. The residue is tested in the DSR and the mass change is checked.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='20' y='12' width='120' height='96' rx='6' fill='#e5e7eb' stroke='#1f2937' stroke-width='2'/><rect x='30' y='22' width='84' height='76' rx='4' fill='#fef3c7' stroke='#1f2937' stroke-width='2'/><circle cx='72' cy='60' r='28' fill='none' stroke='#6b7280' stroke-width='3'/><circle cx='72' cy='60' r='5' fill='#1f2937'/><g fill='#ffffff' stroke='#1f2937' stroke-width='1.5'><rect x='68' y='27' width='8' height='14' rx='2'/><rect x='68' y='79' width='8' height='14' rx='2'/><rect x='39' y='56' width='14' height='8' rx='2'/><rect x='91' y='56' width='14' height='8' rx='2'/><rect x='46' y='36' width='12' height='8' rx='2' transform='rotate(45 52 40)'/><rect x='86' y='76' width='12' height='8' rx='2' transform='rotate(45 92 80)'/><rect x='86' y='36' width='12' height='8' rx='2' transform='rotate(-45 92 40)'/><rect x='46' y='76' width='12' height='8' rx='2' transform='rotate(-45 52 80)'/></g><rect x='120' y='22' width='14' height='30' rx='2' fill='#111827'/><circle cx='127' cy='68' r='4' fill='#e34948'/><circle cx='127' cy='82' r='4' fill='#16a34a'/><path d='M58 22 A26 26 0 0 1 96 30' fill='none' stroke='#e74c3c' stroke-width='2'/><path d='M96 30 l-6 -1 m6 1 l-2 -6' stroke='#e74c3c' stroke-width='2' fill='none'/></svg>"}, {'id': 'pav', 'name': 'Pressure Aging Vessel', 'device': 'Heated pressure vessel with pan rack', 'standard': 'AASHTO R 28', 'stage': 'Long-term ageing', 'color': '#e67e22', 'purpose': 'Simulates several years of ageing in service: the RTFO residue in thin pans is held at high temperature under 2.10 MPa air pressure for 20 hours. The aged binder is then tested in the DSR, BBR and DTT.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='24' y='102' width='112' height='8' rx='3' fill='#e5e7eb' stroke='#1f2937' stroke-width='2'/><rect x='40' y='34' width='80' height='68' rx='10' fill='#9ca3af' stroke='#1f2937' stroke-width='2'/><rect x='36' y='26' width='88' height='12' rx='3' fill='#6b7280' stroke='#1f2937' stroke-width='2'/><g fill='#1f2937'><circle cx='44' cy='32' r='2'/><circle cx='62' cy='32' r='2'/><circle cx='80' cy='32' r='2'/><circle cx='98' cy='32' r='2'/><circle cx='116' cy='32' r='2'/></g><rect x='52' y='46' width='56' height='48' rx='3' fill='#f3f4f6' stroke='#1f2937' stroke-width='1.5'/><g stroke='#1f2937' stroke-width='1.5'><line x1='56' y1='56' x2='104' y2='56'/><line x1='56' y1='68' x2='104' y2='68'/><line x1='56' y1='80' x2='104' y2='80'/></g><g fill='#111827'><rect x='60' y='52' width='40' height='3'/><rect x='60' y='64' width='40' height='3'/><rect x='60' y='76' width='40' height='3'/></g><line x1='80' y1='26' x2='80' y2='16' stroke='#1f2937' stroke-width='3'/><circle cx='80' cy='12' r='10' fill='#ffffff' stroke='#1f2937' stroke-width='2'/><line x1='80' y1='12' x2='86' y2='7' stroke='#e67e22' stroke-width='2'/></svg>"}, {'id': 'bbr', 'name': 'Bending Beam Rheometer', 'device': 'Three-point bending in a cold bath', 'standard': 'AASHTO T 313', 'stage': 'PAV-aged binder, low temperature', 'color': '#2a78d6', 'purpose': 'A small binder beam rests on two supports in a cold liquid bath and a constant load pushes its middle down. The deflection gives the creep stiffness S and its rate of change m, which control thermal cracking.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='18' y='44' width='124' height='64' rx='4' fill='#dbeafe' stroke='#1f2937' stroke-width='2'/><rect x='20' y='56' width='120' height='50' fill='#93c5fd' fill-opacity='0.55'/><path d='M24 58 q8 -4 16 0 t16 0 t16 0 t16 0 t16 0 t16 0 t16 0' fill='none' stroke='#2a78d6' stroke-width='1.5'/><polygon points='42,94 50,94 46,84' fill='#6b7280' stroke='#1f2937' stroke-width='1.5'/><polygon points='110,94 118,94 114,84' fill='#6b7280' stroke='#1f2937' stroke-width='1.5'/><rect x='38' y='78' width='84' height='6' fill='#111827'/><path d='M38 84 Q80 92 122 84' fill='none' stroke='#2a78d6' stroke-width='1.5' stroke-dasharray='3 2'/><rect x='76' y='12' width='8' height='60' fill='#6b7280' stroke='#1f2937' stroke-width='2'/><polygon points='74,72 86,72 80,78' fill='#1f2937'/><path d='M96 18 v24' stroke='#2a78d6' stroke-width='2.5'/><path d='M96 42 l-4 -6 m4 6 l4 -6' stroke='#2a78d6' stroke-width='2.5' fill='none'/><text x='102' y='30' font-size='10' font-family='Arial' fill='#1f2937'>P</text></svg>"}, {'id': 'dtt', 'name': 'Direct Tension Tester', 'device': 'Tension test of a dog-bone specimen', 'standard': 'AASHTO T 314', 'stage': 'PAV-aged binder, low temperature', 'color': '#27ae60', 'purpose': 'A small dog-bone binder specimen is pulled apart at low temperature until it breaks. The failure strain shows whether the binder can stretch enough before cracking; it is used when the BBR stiffness is between 300 and 600 MPa.', 'svg': "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 160 120'><rect x='10' y='22' width='140' height='76' rx='6' fill='none' stroke='#93c5fd' stroke-width='2' stroke-dasharray='5 4'/><rect x='22' y='44' width='22' height='32' rx='3' fill='#6b7280' stroke='#1f2937' stroke-width='2'/><rect x='116' y='44' width='22' height='32' rx='3' fill='#6b7280' stroke='#1f2937' stroke-width='2'/><path d='M44 48 L58 48 Q64 48 66 54 L94 54 Q96 48 102 48 L116 48 L116 72 L102 72 Q96 72 94 66 L66 66 Q64 72 58 72 L44 72 Z' fill='#111827' stroke='#1f2937' stroke-width='1.5'/><path d='M20 60 h-12' stroke='#27ae60' stroke-width='3'/><path d='M8 60 l6 -5 m-6 5 l6 5' stroke='#27ae60' stroke-width='3' fill='none'/><path d='M140 60 h12' stroke='#27ae60' stroke-width='3'/><path d='M152 60 l-6 -5 m6 5 l-6 5' stroke='#27ae60' stroke-width='3' fill='none'/><text x='58' y='92' font-size='10' font-family='Arial' fill='#1f2937'>cold chamber</text></svg>"}]

# Display helpers for the Colab notebook: the three results, styled like the web app.
import base64
import html
import io
import math
from datetime import date

import matplotlib.pyplot as plt
from IPython.display import HTML, display

RED, RED_DARK, BLUE, BLUE_DARK = '#d64541', '#8e1b1b', '#2e6fba', '#123f7a'
MEAN_COLOR, DESIGN_COLOR = '#2e7d32', '#111827'

BASE_CSS = """
<style>
.pgx{font-family:Inter,Segoe UI,Arial,sans-serif;color:#1f2937;max-width:980px}
.pgx h2{font-size:20px;margin:28px 0 10px;padding-bottom:6px;border-bottom:3px solid var(--c)}
.pgx table.data{border-collapse:collapse;font-size:13px;margin:6px 0 12px}
.pgx table.data th{background:var(--c);color:#fff;padding:6px 9px;font-weight:600;text-align:center}
.pgx table.data td{border:1px solid #e5e7eb;padding:5px 9px;text-align:center;font-variant-numeric:tabular-nums}
.pgx table.data tr:nth-child(even) td{background:#f9fafb}
.pgx table.data tr.hl td{background:var(--hl);font-weight:700}
.pgx table.data tr.off td{background:#f3f4f6;color:#9ca3af;font-style:italic}
.pgx table.data tr.off td.why{text-align:left}
.pgx .summary{display:flex;gap:12px;flex-wrap:wrap;margin:6px 0 4px}
.pgx .summary div{border:1px solid #e5e7eb;border-left:4px solid var(--c);border-radius:6px;padding:8px 14px;background:#fff}
.pgx .summary b{display:block;font-size:18px}
.pgx .muted{color:#6b7280;font-size:12px}
.pgx table.heat{border-collapse:separate;border-spacing:2px;font-size:11px;margin-top:6px}
.pgx table.heat td{width:44px;height:22px;text-align:center;border-radius:3px;font-variant-numeric:tabular-nums}
.pgx table.heat th{font-weight:600;color:#6b7280;padding:0 6px;text-align:left}
.pgx table.heat tr.all td{color:#374151}
.pgx .error{border:1px solid #fca5a5;background:#fef2f2;color:#991b1b;padding:14px 18px;border-radius:8px;font-size:15px}
.pgx .card{border-radius:12px;padding:18px 22px;background:linear-gradient(135deg,#0f766e,#1d4ed8);color:#fff;margin:10px 0 16px}
.pgx .card .big{font-size:28px;font-weight:800;letter-spacing:.5px}
.pgx .card .sub{opacity:.92;margin-top:6px;font-size:14px}
.pgx .head{display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:6px;
  border-bottom:1px solid #e5e7eb;padding-bottom:8px}
.pgx .head .t{font-size:22px;font-weight:800}
table.spec{border-collapse:collapse;width:100%;max-width:760px;font-size:13px;font-family:Arial,Helvetica,sans-serif}
table.spec td{border:1px solid #c9ced6;padding:7px 9px}
table.spec td.c{text-align:center}
.spec-title{font-family:'Times New Roman',serif;font-weight:700;font-size:17px;margin:14px 0 8px}
.spec-note{font-family:'Times New Roman',serif;font-size:14px;max-width:760px;text-align:justify;margin-top:10px}
</style>
"""


def show_html(body):
    display(HTML(BASE_CSS + body))


def show_error(message):
    show_html(f'<div class="pgx"><div class="error">⚠️ {html.escape(message)}</div></div>')


def _fmt_date(iso):
    d = date.fromisoformat(iso)
    return d.strftime('%d %b %Y').lstrip('0')


def _f(v, n=1):
    return f'{v:.{n}f}'.replace('-', '−')


def _png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=160, bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def bar_chart(years, values, highlight_year, mean_value, design_value, color, color_dark, ylabel, title, decimals):
    fig, ax = plt.subplots(figsize=(10, 4.3))
    colors = [color_dark if y == highlight_year else color for y in years]
    bars = ax.bar([str(y) for y in years], values, color=colors, edgecolor='white', linewidth=0.8, zorder=3)
    if design_value is None:
        ax.axhline(mean_value, color=DESIGN_COLOR, linestyle='-', linewidth=2, zorder=4,
                   label=f'Design (mean): {mean_value:.2f} °C')
        lines = [mean_value]
    else:
        ax.axhline(mean_value, color=MEAN_COLOR, linestyle='--', linewidth=1.8, zorder=4,
                   label=f'Mean: {mean_value:.{decimals}f} °C')
        ax.axhline(design_value, color=DESIGN_COLOR, linestyle='-', linewidth=2, zorder=4,
                   label=f'Design (98%): {design_value:.{decimals}f} °C')
        lines = [mean_value, design_value]
    lo = min(values + lines)
    hi = max(values + lines)
    pad = max(0.8, (hi - lo) * 0.25)
    ax.set_ylim(lo - pad, hi + pad)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + (hi - lo + 2 * pad) * 0.01, f'{v:.{decimals}f}',
                ha='center', va='bottom', fontsize=8, zorder=5)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), frameon=False)
    plt.setp(ax.get_xticklabels(), rotation=45)
    return _png(fig)


def show_header(project_name, station_name, latitude, grade):
    station = f' · Station: {html.escape(station_name)}' if station_name else ''
    show_html(f"""<div class="pgx"><div class="head">
      <div><div class="t">Superpave PG Binder Selector</div>
      <div class="muted">Project: {html.escape(project_name)}{station} · Latitude: {latitude:g}° ·
      Reliability: {grade['reliabilityPercent']}%</div></div>
      <div class="muted">Prepared by Abenezer Tilahun</div></div></div>""")


def _why_not_used(e, high):
    if e['reason'] == 'no-usable-period':
        return 'Not used: no 7 recorded days in a row' if high else 'Not used: no usable day'
    if e['realFraction'] == 0:
        return 'Not used: no data this year'
    pct = math.floor(e['realFraction'] * 1000) / 10
    return f'Not used: only {pct:g}% of days recorded (at least {MIN_REAL_FRACTION * 100:g}% needed)'


def _with_left_out_rows(used_rows, excluded, colspan, high):
    """All years in order: used rows as given, left-out years greyed with the reason."""
    rows = dict(used_rows)
    for e in excluded:
        rows[e['year']] = (f"<tr class='off'><td>{e['year']}</td>"
                           f"<td class='why' colspan='{colspan}'>{_why_not_used(e, high)}</td></tr>")
    return ''.join(rows[y] for y in sorted(rows))


def _left_out_note(excluded):
    if not excluded:
        return ''
    return (f' Grey rows: {len(excluded)} year{"s" if len(excluded) > 1 else ""} left out because less than {MIN_REAL_FRACTION * 100:g}% '
            f'of the days were recorded; they are not used in the graph or the design.')


def show_seven_day(climate, grade):
    years = climate['high']['years']
    hottest = grade['high']['hottestYear']
    used = [(y['year'],
             f"<tr class='{'hl' if y['year'] == hottest else ''}'><td>{y['year']}</td><td>{_fmt_date(y['startDate'])}</td>"
             f"<td>{_fmt_date(y['endDate'])}</td>" + ''.join(f'<td>{_f(d)}</td>' for d in y['days'])
             + f"<td><b>{_f(y['average'], 2)}</b></td></tr>") for y in years]
    rows = _with_left_out_rows(used, climate['high'].get('excluded', []), 10, True)
    head = ''.join(f'<th>Day {i}</th>' for i in range(1, 8))
    chart = bar_chart([y['year'] for y in years], [y['average'] for y in years], hottest, grade['high']['mean'],
                      grade['high']['designAirTemp'], RED, RED_DARK, '7-day average (°C)',
                      'Annual 7-day consecutive maximum air temperature', 2)
    h = grade['high']
    show_html(f"""<div class="pgx" style="--c:{RED};--hl:#fde2e1">
      <h2>1 · Annual 7-Day Consecutive Maximum Temperature</h2>
      <table class="data"><tr><th>Year</th><th>Start date</th><th>End date</th>{head}<th>7-day average (°C)</th></tr>
      {rows}</table>
      <div class="muted">The hottest year ({hottest}) is highlighted. Years used: {h['yearsUsed']}.{_left_out_note(climate['high'].get('excluded', []))}</div>
      <img style="max-width:100%;margin-top:8px" src="data:image/png;base64,{chart}"/>
      <div class="summary"><div>Mean<b>{_f(h['mean'], 2)} °C</b></div>
      <div>Standard deviation<b>{_f(h['sd'], 2)} °C</b></div>
      <div>Design high air temperature (98%) = mean + 2.054 σ<b>{_f(h['designAirTemp'], 2)} °C</b></div></div></div>""")
    return chart


def show_annual_min(climate, grade):
    years = climate['low']['years']
    coldest = grade['low']['coldestYear']
    used = [(y['year'], f"<tr class='{'hl' if y['year'] == coldest else ''}'><td>{y['year']}</td>"
                        f"<td>{_fmt_date(y['date'])}</td><td><b>{_f(y['value'])}</b></td></tr>") for y in years]
    rows = _with_left_out_rows(used, climate['low'].get('excluded', []), 2, False)
    chart = bar_chart([y['year'] for y in years], [y['value'] for y in years], coldest, grade['low']['mean'],
                      None, BLUE, BLUE_DARK, 'Minimum temperature (°C)',
                      'Annual minimum air temperature', 1)
    lo = grade['low']
    show_html(f"""<div class="pgx" style="--c:{BLUE};--hl:#dbeafe">
      <h2>2 · Annual Minimum Temperature</h2>
      <table class="data"><tr><th>Year</th><th>Date</th><th>Minimum temperature (°C)</th></tr>{rows}</table>
      <div class="muted">The coldest year ({coldest}) is highlighted. Years used: {lo['yearsUsed']}.{_left_out_note(climate['low'].get('excluded', []))}</div>
      <img style="max-width:100%;margin-top:8px" src="data:image/png;base64,{chart}"/>
      <div class="summary"><div>Mean<b>{_f(lo['mean'], 2)} °C</b></div>
      <div>Standard deviation<b>{_f(lo['sd'], 2)} °C</b></div>
      <div>Low air temperature used in design (mean)<b>{_f(lo['designAirTemp'], 2)} °C</b></div></div></div>""")
    return chart


RED_RAMP = ['#fde4e3', '#fbc9c7', '#f7a8a6', '#f08785', '#e86866', '#e34948', '#c63a3a', '#a12e2e', '#7a2222']
BLUE_RAMP = ['#cde2fb', '#9ec5f4', '#86b6ef', '#6da7ec', '#3987e5', '#2a78d6', '#1c5cab', '#184f95', '#0d366b']
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


def _heatmap_html(title, cells, ramp, colder_is_darker, marked, not_used):
    means = [c['mean'] for c in cells if c['mean'] is not None]
    lo, hi = min(means), max(means)
    years = sorted({c['year'] for c in cells})
    grid = {(c['year'], c['month']): c for c in cells}
    rows = []
    for y in years:
        tds = []
        for m in range(1, 13):
            c = grid.get((y, m))
            if c is None or c['mean'] is None:
                tds.append('<td style="background:repeating-linear-gradient(135deg,#e5e7eb 0 2px,#fff 2px 7px)"></td>')
                continue
            t = 0.5 if hi == lo else (c['mean'] - lo) / (hi - lo)
            t = 1 - t if colder_is_darker else t
            i = max(0, min(len(ramp) - 1, round(t * (len(ramp) - 1))))
            color = '#fff' if i >= (len(ramp) + 1) // 2 else '#111'
            ring = 'outline:2px solid #111;outline-offset:-2px;' if marked.get(y) == m else ''
            tds.append(f'<td style="background:{ramp[i]};color:{color};{ring}">{_f(c["mean"])}</td>')
        label = f'<i style="color:#9ca3af">{y} ✕</i>' if y in not_used else str(y)
        rows.append(f'<tr><th>{label}</th>{"".join(tds)}</tr>')
    month_means = []
    for m in range(1, 13):
        v = [c['mean'] for c in cells if c['month'] == m and c['mean'] is not None]
        month_means.append(f'<td><b>{_f(sum(v) / len(v)) if v else "—"}</b></td>')
    legend = ramp[::-1] if colder_is_darker else ramp
    return (f"<div style='margin:10px 0 18px'><div style='display:flex;justify-content:space-between;align-items:center'>"
            f"<b>{title}</b><span class='muted'>{_f(lo)} <span style='display:inline-block;width:110px;height:10px;"
            f"border-radius:6px;background:linear-gradient(90deg,{','.join(legend)})'></span> {_f(hi)} °C</span></div>"
            f"<table class='heat'><tr><th></th>{''.join(f'<th>{m}</th>' for m in MONTHS)}</tr>{''.join(rows)}"
            f"<tr class='all'><th>All yrs</th>{''.join(month_means)}</tr></table></div>")


def show_heatmaps(climate):
    hot = {y['year']: int(y['startDate'][5:7]) for y in climate['high']['years']}
    cold = {y['year']: int(y['date'][5:7]) for y in climate['low']['years']}
    show_html(f"""<div class="pgx" style="--c:#374151"><h2>3 · Monthly Temperature Heatmaps</h2>
      <div class="muted">Mean of the recorded daily values for each month of each year (filled gaps are not shown).</div>
      {_heatmap_html('Daily maximum temperature (Tmax), monthly mean', climate['monthly']['tmax'], RED_RAMP, False, hot,
                     set(climate['high']['excludedYears']))}
      {_heatmap_html('Daily minimum temperature (Tmin), monthly mean', climate['monthly']['tmin'], BLUE_RAMP, True, cold,
                     set(climate['low']['excludedYears']))}
      <div class="muted">Outlined: month of the year's hottest 7-day window (Tmax) or coldest night (Tmin).
      Hatched: fewer than half of the month's days recorded. ✕: year not used in the design.</div></div>""")


BINDER_COLORS = {1: '#c6e3b3', 2: '#fff34d', 3: '#d39fd6'}


def _lab_requirements(spec):
    L = spec['limits']
    sel = spec['columns'][spec['selectedIndex']]
    return {
        'flash': [f"Flash point ≥ {L['flashPointMinC']} °C"],
        'rv': [f"Viscosity ≤ {L['viscosityMaxPaS']} Pa·s at {L['viscosityTestTempC']} °C"],
        'dsr': [f"Original binder: G*/sin δ ≥ {L['originalDsrMinKPa']:.2f} kPa at {spec['originalDsrTempC']} °C",
                f"RTFO residue: G*/sin δ ≥ {L['rtfoDsrMinKPa']:.2f} kPa at {spec['rtfoDsrTempC']} °C",
                f"PAV residue: G*·sin δ ≤ {L['pavDsrMaxKPa']} kPa at {_f(sel['pavDsrTempC'], 0)} °C"],
        'rtfo': [f"Mass change ≤ {L['rtfoMassChangeMaxPct']:.2f}%", '163 °C, 85 minutes'],
        'pav': [f"{spec['pavAgingTempLabel']} °C, {L['pavPressureMPa']:.2f} MPa, {L['pavAgingHours']} hours"],
        'bbr': [f"S ≤ {L['bbrStiffnessMaxMPa']} MPa and m ≥ {L['bbrMValueMin']:.3f} at {_f(sel['bbrTempC'], 0)} °C"],
        'dtt': [f"Failure strain ≥ {L['dttFailureStrainMinPct']:.2f}% at {_f(sel['dttTempC'], 0)} °C"],
    }


def show_binder(grade):
    """Section 5: binder type chart (selected grade marked) and the laboratory tests for the grade."""
    bt = grade['binderType']
    H, L = grade['finalGrade']['high'], grade['finalGrade']['low']
    head = ''.join(f"<th style='padding:4px 8px;color:{'#111' if h == H else '#6b7280'}'>{h}</th>" for h in BINDER_MATRIX_HIGH)
    rows = []
    for i, low in enumerate(BINDER_MATRIX_LOW):
        cells = []
        for h in BINDER_MATRIX_HIGH:
            here = h == H and low == L
            mark = '<div style="font-size:9px;font-weight:700">YOUR GRADE</div>' if here else ''
            ring = 'outline:3px solid #111;outline-offset:-3px;font-weight:700;' if here else ''
            cells.append(f"<td style='background:{BINDER_COLORS[BINDER_MATRIX[h][i]]};{ring}padding:8px 10px;text-align:center;"
                         f"border-radius:5px;color:#1f2937'>{h}{low}{mark}</td>")
        rows.append(f"<tr><th style='padding:4px 8px;color:{'#111' if low == L else '#6b7280'}'>{_f(low, 0)}</th>{''.join(cells)}</tr>")
    legend = ''.join(f"<span style='margin-right:16px'><span style='display:inline-block;width:12px;height:12px;border-radius:3px;"
                     f"background:{BINDER_COLORS[c]};vertical-align:middle;margin-right:5px'></span>{BINDER_TYPES[c][0]}</span>"
                     for c in (1, 2, 3))
    rule = f"{H} + {abs(L)} = {bt['temperatureSum']} {'&gt; 90' if bt['ruleOf90SaysModified'] else '≤ 90'}"
    reqs = _lab_requirements(grade['spec'])
    cards = []
    for t in LAB_EQUIPMENT:
        items = ''.join(f"<li style='background:#f4f4f5;border-radius:4px;padding:2px 6px;margin-top:3px'>{r}</li>"
                        for r in reqs.get(t['id'], []))
        cards.append(f"<div style='border:1px solid #e5e7eb;border-top:4px solid {t['color']};border-radius:10px;overflow:hidden;background:#fff'>"
                     f"<div style='height:120px;display:flex;align-items:center;justify-content:center;padding:6px'>"
                     f"<div style='width:160px;height:120px'>{t['svg']}</div></div><div style='padding:8px 12px 12px'>"
                     f"<b>{t['name']}</b> <span class='muted'>{t['standard']}</span>"
                     f"<div style='color:{t['color']};font-size:12px;font-weight:600'>{t['stage']}</div>"
                     f"<div class='muted' style='font-size:12.5px;margin-top:4px'>{t['purpose']}</div>"
                     f"<ul style='list-style:none;padding:0;margin:6px 0 0;font-size:12.5px;font-weight:600'>{items}</ul></div></div>")
    show_html(f"""<div class="pgx" style="--c:#374151"><h2>5 · Binder Type and Laboratory Tests</h2>
      <div style='display:flex;align-items:center;gap:12px;border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;
        background:{BINDER_COLORS[bt['code']]}55;margin-bottom:10px'>
        <span style='width:30px;height:30px;border-radius:6px;background:{BINDER_COLORS[bt['code']]};border:1px solid #9ca3af'></span>
        <div><div class='muted'>BINDER TYPE FOR {grade['finalGrade']['label']}</div><b style='font-size:18px'>{bt['name']}</b>
        <div class='muted'>{bt['description']}</div></div>
        <div style='margin-left:auto;text-align:right'><div class='muted'>RULE OF 90</div><b>{rule}</b>
        <div class='muted'>{'likely modified' if bt['ruleOf90SaysModified'] else 'typically unmodified'}</div></div></div>
      <table style='border-collapse:separate;border-spacing:3px'><tr><th class='muted'>Low \ High</th>{head}</tr>{''.join(rows)}</table>
      <div style='margin:8px 0 18px;font-size:13px'>{legend}</div>
      <b>Laboratory tests and equipment for {grade['finalGrade']['label']}</b>
      <div style='display:grid;grid-template-columns:repeat(auto-fill,minmax(230px,1fr));gap:12px;margin-top:8px'>{''.join(cards)}</div>
      </div>""")


ESAL_LABELS = {'lt0.3': '< 0.3', '0.3to3': '0.3 to < 3', '3to10': '3 to < 10', '10to30': '10 to < 30', 'ge30': '≥ 30'}
SPEED_LABELS = {'standing': 'standing', 'slow': 'slow', 'standard': 'standard'}


def models_html(grade, latitude):
    """SHRP (high) and LTPP (low) design temperature models: formula, then the project's numbers."""
    lat = abs(latitude)
    lat2 = lat * lat
    z, h, l = grade['z'], grade['high'], grade['low']
    high_sum = h['designAirTemp'] - 0.00618 * lat2 + 0.2289 * lat + 42.2
    root = math.sqrt(4.4 + 0.52 * l['sd'] ** 2)
    f = lambda x, d=2: _f(x, d)  # noqa: E731

    def row(label, body, strong=False):
        weight = 'font-weight:700;' if strong else ''
        return (f"<tr><td style='width:96px;color:#6b7280;font-size:11px;text-transform:uppercase;letter-spacing:.04em'>"
                f"{label}</td><td style=\"font-family:'Times New Roman',serif;font-size:15px;{weight}\">{body}</td></tr>")

    def card(color, tag, title, sub, rows):
        return (f"<div style='border:1px solid #e5e7eb;border-top:4px solid {color};border-radius:10px;padding:12px 16px;"
                f"background:#f9fafb;margin:10px 0'><div style='display:flex;justify-content:space-between;flex-wrap:wrap'>"
                f"<b><span style='background:{color};color:#fff;border-radius:5px;padding:1px 7px;margin-right:6px'>{tag}</span>"
                f"{title}</b><span class='muted'>{sub}</span></div><table style='margin-top:6px'>{''.join(rows)}</table></div>")

    bump = (f" → traffic → <span style='color:{RED}'>PG {grade['finalGrade']['high']}</span>"
            if grade['finalGrade']['high'] != grade['climateGrade']['high'] else '')
    shrp = card(RED, 'SHRP', 'High-temperature model', 'Pavement temperature at 20 mm depth', [
        row('Formula', 'T<sub>air</sub> = T̄<sub>7-day</sub> + z·σ<sub>7-day</sub>'),
        row('', 'T<sub>20mm</sub> = (T<sub>air</sub> − 0.00618·Lat² + 0.2289·Lat + 42.2) × 0.9545 − 17.78'),
        row('Inputs', f"T̄<sub>7-day</sub> = {f(h['mean'])} °C, σ = {f(h['sd'])} °C, z = {f(z, 3)} (98%), Lat = {f(lat)}° "
                      f"({h['yearsUsed']} years)"),
        row('Air', f"T<sub>air</sub> = {f(h['mean'])} + {f(z, 3)} × {f(h['sd'])} = <b>{f(h['designAirTemp'])} °C</b>"),
        row('Pavement', f"T<sub>20mm</sub> = ({f(h['designAirTemp'])} − {f(0.00618 * lat2, 3)} + {f(0.2289 * lat, 3)} + 42.2)"
                        f" × 0.9545 − 17.78"),
        row('', f"= {f(high_sum, 3)} × 0.9545 − 17.78 = <b>{f(h['pavementTemp'])} °C</b>"),
        row('Grade', f"{f(h['pavementTemp'])} °C → rounded up to <span style='color:{RED}'>PG {grade['climateGrade']['high']}"
                     f"</span>{bump}", True),
    ])
    ltpp = card(BLUE, 'LTPP', 'Low-temperature model', 'Pavement temperature at the surface (H = 0 mm)', [
        row('Formula', 'T<sub>pav</sub> = −1.56 + 0.72·T<sub>air</sub> − 0.004·Lat² + 6.26·log<sub>10</sub>(H + 25) − '
                       'z·√(4.4 + 0.52·σ²)'),
        row('Inputs', f"T<sub>air</sub> = {f(l['meanAirTemp'])} °C (mean of yearly minima), σ = {f(l['sd'])} °C, "
                      f"z = {f(z, 3)}, Lat = {f(lat)}°, H = 0 ({l['yearsUsed']} years)"),
        row('Terms', f"−1.56 + {f(0.72 * l['meanAirTemp'], 3)} − {f(0.004 * lat2, 3)} + {f(6.26 * math.log10(25), 3)} = "
                     f"{f(l['baseTemp'], 3)}"),
        row('Reliability', f"{f(z, 3)} × √(4.4 + 0.52 × {f(l['sd'])}²) = {f(z, 3)} × {f(root, 3)} = "
                           f"{f(l['reliabilityTerm'], 3)}"),
        row('Pavement', f"T<sub>pav</sub> = {f(l['baseTemp'], 3)} − {f(l['reliabilityTerm'], 3)} = "
                        f"<b>{f(l['pavementTemp'])} °C</b>"),
        row('Grade', f"{f(l['pavementTemp'])} °C → rounded down to <span style='color:{BLUE}'>"
                     f"{_f(grade['climateGrade']['low'], 0)}</span>", True),
    ])
    return shrp + ltpp


def show_grade(grade, inputs, spec_table_html):
    t = grade['traffic']
    if t['isConsiderCase']:
        bump = ('+1 grade (you chose to add the grade AASHTO M323 says to consider)' if t['gradesAdded']
                else 'no change. AASHTO M323 says to consider +1 grade: tick "add_one_grade_if_considered" in '
                     'Step 1 and run Step 3 again to add it')
    else:
        bump = 'no change' if t['gradesAdded'] == 0 else f"+{t['gradesAdded']} grade{'s' if t['gradesAdded'] > 1 else ''}"
    capped = '<div class="sub">⚠️ Limited to PG 88, the highest grade.</div>' if t['cappedAtMax'] else ''
    out_of_range = ''
    if grade['high']['outOfRange']:
        out_of_range += '<div class="sub">⚠️ The high pavement temperature is above 88 °C.</div>'
    if grade['low']['outOfRange']:
        out_of_range += '<div class="sub">⚠️ The low pavement temperature is below −46 °C.</div>'
    show_html(f"""<div class="pgx" style="--c:#0f766e"><h2>4 · Final PG Grade and Specification</h2>
      <div class="card"><div class="big">Reliability: {grade['reliabilityPercent']}% &nbsp;|&nbsp;
      {grade['climateGrade']['label']} → {grade['finalGrade']['label']}</div>
      <div class="sub">Climate: SHRP T<sub>20 mm</sub> = {_f(grade['high']['pavementTemp'], 2)} °C → PG {grade['climateGrade']['high']},
      &nbsp; LTPP T<sub>pav,low</sub> = {_f(grade['low']['pavementTemp'], 2)} °C → {_f(grade['climateGrade']['low'], 0)}</div>
      <div class="sub">Traffic: {inputs['designEsalMillions']:g} million ESAL ({ESAL_LABELS[t['esalClass']]}),
      {inputs['designSpeedKmh']:g} km/h ({SPEED_LABELS[t['speedClass']]}) → {bump}</div>{capped}{out_of_range}</div>
      {models_html(grade, inputs['latitude'])}
      {spec_table_html}</div>""")


# Section colours and icons as in the reference table.
C = {'blue': '#2b8bd8', 'teal': '#17a589', 'red': '#ff6b6b', 'pink': '#fde4e4', 'lblue': '#dbeafa',
     'orange': '#f39c12', 'purple': '#8e44ad', 'rtfo': '#e74c3c', 'pav': '#e67e22', 'yellow': '#fef3cd',
     'green_l': '#d9f2e3', 'lav': '#dfe6fb', 'dtt': '#27ae60', 'white': '#ffffff'}


def spec_table_html(spec):
    cols = spec['columns']
    n = len(cols)
    L = spec['limits']

    def td(text, bg, colspan=1, bold=False, color='#1f2937', center=True, extra=''):
        style = f'background:{bg};color:{color};{"font-weight:700;" if bold else ""}{extra}'
        cls = ' class="c"' if center else ''
        span = f' colspan="{colspan}"' if colspan > 1 else ''
        return f'<td{cls}{span} style="{style}">{text}</td>'

    def section(title, bg):
        return f'<tr>{td(title, bg, n + 1, True, "#fff", center=False)}</tr>'

    def per_column(values, bg):
        return ''.join(td(_f(v, 0), C['orange'] if c['selected'] else bg, bold=c['selected'],
                          color='#fff' if c['selected'] else '#1f2937') for c, v in zip(cols, values))

    def row(left, right_cells, left_bg=C['white']):
        return f'<tr>{td(left, left_bg, center=False)}{right_cells}</tr>'

    hg = spec['highGrade']
    rows = [
        f"<tr>{td('Performance Grades', C['blue'], 2, True, '#fff')}{td('', C['blue'], n - 1)}</tr>",
        f"<tr>{td('Property', C['teal'], 1, True, '#fff', center=False)}{td(f'PG {hg}', C['teal'], n, True, '#fff')}</tr>",
        f"<tr>{td('🌡️ <b>Max. Design Temp. (°C)</b>', C['pink'], center=False, color='#c0392b')}"
        f"{td(str(hg), C['red'], n, True, '#fff')}</tr>",
        row('❄️ <b>Min. Design Temp. (°C)</b>', per_column([c['lowGrade'] for c in cols], C['lblue']), C['lblue']),
        section('🧪 Original Binder', C['purple']),
        row(f"≥{L['flashPointMinC']} °C", td('🔥 Flash Point', C['yellow'], n, center=False)),
        row(f"≤{L['viscosityMaxPaS']} Pa·s @ {L['viscosityTestTempC']} °C",
            td('💧 Rotational Viscosity', C['lblue'], n, center=False)),
        row(f"≥{L['originalDsrMinKPa']:.2f} kPa", td('⚙️ DSR G*/sin δ (Dynamic Shear Rheometer)', C['green_l'], n, center=False)),
        row('', td(str(spec['originalDsrTempC']), C['red'], n, True, '#fff')),
        section(f"🔥 (Rolling Thin Film Oven) RTFO, Mass Change ≤ {L['rtfoMassChangeMaxPct']:.2f}%", C['rtfo']),
        row(f"≥{L['rtfoDsrMinKPa']:.2f} kPa", td('⚙️ DSR G*/sin δ (Dynamic Shear Rheometer)', C['green_l'], n, center=False)),
        row('', td(str(spec['rtfoDsrTempC']), C['red'], n, True, '#fff')),
        section('🏺 (Pressure Aging Vessel) PAV', C['pav']),
        row(f"{L['pavAgingHours']} hours, {L['pavPressureMPa']:.2f} MPa",
            td(spec['pavAgingTempLabel'].replace(' (', '('), C['yellow'], n, True), C['yellow']),
        row(f"≤{L['pavDsrMaxKPa']} kPa", td('⚙️ DSR G*·sin δ (Dynamic Shear Rheometer) intermediate Temp = '
                                             '[(Max. + Min.)/2] + 4', C['lblue'], n, center=False)),
        row('', per_column([c['pavDsrTempC'] for c in cols], C['lblue'])),
        row(f"S ≤ {L['bbrStiffnessMaxMPa']} MPa<br>m ≥ {L['bbrMValueMin']:.3f}",
            td('📊 BBR S (creep stiffness) &amp; m-value (Bending Beam Rheometer)', C['lav'], n, center=False)),
        row('', per_column([c['bbrTempC'] for c in cols], C['lav'])),
        section('📏 DTT (Direct Tension Tester)', C['dtt']),
        row(f"ε<sub>f</sub> ≥ {L['dttFailureStrainMinPct']:.2f}%", per_column([c['dttTempC'] for c in cols], C['white'])),
    ]
    widths = '<col style="width:24%">' + ''.join(f'<col style="width:{76 / n:.2f}%">' for _ in cols)
    notes = ''
    if not spec['inM320']:
        notes += (f"<p class='spec-note'><i>{spec['label']} is not listed in AASHTO M320 Table 1; the test "
                  f"temperatures above follow the same rules.</i></p>")
    return (f"<div class='spec-title'>{html.escape(spec['title'])}</div>"
            f"<table class='spec'><colgroup>{widths}</colgroup>{''.join(rows)}</table>"
            f"<p class='spec-note'><b>Note:</b> <i><b>{html.escape(NOTE)}</b></i></p>"
            f"<p class='spec-note' style='font-size:12px;color:#4b5563'>DTT: {html.escape(DTT_NOTE)}</p>{notes}")


NOTE = ('The highlighted column (Orange background) indicates the test temperatures corresponding to the project '
        'selected PG grade. All values must meet the specified criteria for the binder to be certified for this '
        'performance grade according to AASHTO M320/M332 standards.')
DTT_NOTE = ('under AASHTO M320 the direct tension test is needed only when the BBR creep stiffness S is between 300 '
            'and 600 MPa (with m ≥ 0.300).')

print('✓ Engine ' + ENGINE_VERSION + ' loaded. Next: fill in Step 1.')


In [ ]:
#@title Step 1 – Project inputs { display-mode: "form" }
project_name = "My project" #@param {type:"string"}
#@markdown Leave empty to use the station name in the file.
station_name = "" #@param {type:"string"}
#@markdown Degrees, north positive. Leave empty to use the latitude in the file.
latitude = "" #@param {type:"string"}
design_esal_millions = 115.6 #@param {type:"number"}
design_speed_kmh = 100 #@param {type:"number"}
#@markdown AASHTO M323 says "consider +1 grade" for 10–30 million ESAL at more than 70 km/h, and for less than 0.3 million ESAL with standing traffic. Tick to add that grade.
add_one_grade_if_considered = False #@param {type:"boolean"}
print('✓ Inputs saved. Next: run Step 2.')


In [ ]:
#@title Step 2 – Choose the daily temperature file (CSV or Excel) { display-mode: "form" }
#@markdown Outside Google Colab, type the full path of the file here instead:
file_path = "" #@param {type:"string"}
if file_path.strip():
    with open(file_path.strip(), 'rb') as _f:
        file_bytes = _f.read()
    file_name = os.path.basename(file_path.strip())
else:
    from google.colab import files as _colab_files
    _uploaded = _colab_files.upload()
    if _uploaded:
        file_name, file_bytes = list(_uploaded.items())[-1]
if 'file_bytes' in globals():
    print(f'✓ {file_name} loaded ({len(file_bytes) / 1024:.0f} KB). Next: run Step 3.')


In [ ]:
#@title Step 3 – Results { display-mode: "form" }
def run_step3():
    global last_charts
    if 'file_bytes' not in globals():
        return show_error('Run Step 2 first and choose your temperature file.')
    if not str(project_name).strip():
        return show_error('Enter a project name.')
    try:
        parsed = read_temperature_file(file_bytes, file_name)
        lat_text = str(latitude).strip()
        if lat_text:
            lat = to_number(lat_text)
            if lat is None:
                raise AnalysisError('INVALID_INPUT', 'Latitude must be a number from -90 to 90°.')
        else:
            lat = parsed['station']['latitude']
            if lat is None:
                raise AnalysisError('INVALID_INPUT', 'Enter the latitude of the station in Step 1.')
        inputs = {'latitude': float(lat), 'designEsalMillions': float(design_esal_millions),
                  'designSpeedKmh': float(design_speed_kmh),
                  'acceptConsiderBump': bool(add_one_grade_if_considered)}
        climate, grade = analyse(parsed['observations'], inputs)
    except AnalysisError as e:
        return show_error(e.message)
    station = str(station_name).strip() or parsed['station']['name'] or ''
    show_header(str(project_name).strip(), station, inputs['latitude'], grade)
    chart_high = show_seven_day(climate, grade)
    chart_low = show_annual_min(climate, grade)
    show_heatmaps(climate)
    show_grade(grade, inputs, spec_table_html(grade['spec']))
    show_binder(grade)
    last_charts = {'7day_maximum.png': chart_high, 'annual_minimum.png': chart_low}

run_step3()


---
**Optional:** download the graphs or an Excel template.

In [ ]:
#@title Optional – download the two graphs as PNG { display-mode: "form" }
download_graphs = False #@param {type:"boolean"}
if not download_graphs:
    print('Tick "download_graphs" and run this cell to download the two graphs.')
elif 'last_charts' not in globals():
    print('Run Step 3 first.')
else:
    import base64 as _b64
    from google.colab import files as _colab_files
    for _name, _data in last_charts.items():
        with open(_name, 'wb') as _f:
            _f.write(_b64.b64decode(_data))
        _colab_files.download(_name)


In [ ]:
#@title Optional – download an Excel template { display-mode: "form" }
download_template = False #@param {type:"boolean"}
if not download_template:
    print('Tick "download_template" and run this cell to download an Excel template.')
else:
    import openpyxl as _xl
    from google.colab import files as _colab_files
    _wb = _xl.Workbook()
    _ws = _wb.active
    _ws.title = 'Data'
    _ws.append(['Name', 'Latitude', 'element', 'Year', 'Month'] + list(range(1, 32)))
    _ex = [27, 28, 31.2, 29.2, 30.2, 30.2, 28.2, 28, 27.5, 28.4, 28.8, 27.5, 28.8, 27.8, 27.5, 30.5, 29.6, 29.7, 32.6,
           30, 29.2, 32, 31, 31, 29.5, 29, 29.5, 30.5, 33.8, 28.5, 27.5]
    _ws.append(['My Station', 9.23, 'TMPMAX', 2024, 1] + _ex)
    _ws.append(['My Station', 9.23, 'TMPMIN', 2024, 1] + [round(v - 15.5, 1) for v in _ex])
    _help = _wb.create_sheet('Instructions')
    for _line in ['Fill the sheet "Data": one row per station, element, year and month.',
                  'element: TMPMAX for daily maximum, TMPMIN for daily minimum temperature (°C).',
                  'Columns 1 … 31: the value of each day. Leave a cell empty (or write X) when there is no reading.',
                  'Other layouts also work, e.g. Date | Tmax | Tmin (one row per day).',
                  'Use at least 5 years (10 or more is better). A year is used only if 75% of its days have readings.']:
        _help.append([_line])
    _wb.save('PG_temperature_template.xlsx')
    _colab_files.download('PG_temperature_template.xlsx')
